In [2]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
# 1. Import the built-in in-memory checkpointer
from langgraph.checkpoint.memory import MemorySaver

# Define our state
class ChatState(TypedDict):
    user_name: str
    chat_history: Annotated[list, operator.add]

# Define a simple node
def greeting_node(state: ChatState):
    history = state["chat_history"]
    if not history:
        reply = f"Hello {state['user_name']}! Nice to meet you."
    else:
        reply = f"Welcome back, {state['user_name']}! You have sent {len(history) + 1} messages total."
    
    return {"chat_history": [reply]}

# Build the graph
builder = StateGraph(ChatState)
builder.add_node("greeter", greeting_node)
builder.add_edge(START, "greeter")
builder.add_edge("greeter", END)

# =========================================================
# THE TRANSFORMATION: Add Memory during Compilation
# =========================================================
memory_storage = MemorySaver()

# We pass the checkpointer straight into the compiler!
persistent_app = builder.compile(checkpointer=memory_storage)
print("💾 Persistent Graph Compiled with Memory Engine!")

💾 Persistent Graph Compiled with Memory Engine!


In [3]:
# --- SESSION 1: Meeting Abhishek ---
config_1 = {"configurable": {"thread_id": "abhishek_session"}}

print("--- RUN 1 ---")
output1 = persistent_app.invoke({"user_name": "Abhishek", "chat_history": ["Message 1"]}, config=config_1)
print(f"Agent says: {output1['chat_history'][-1]}")

# --- SESSION 2: Running it again on the same thread ---
print("\n--- RUN 2 (Same Thread) ---")
# Notice we don't even need to pass the "user_name" anymore! It's saved in the checkpoint.
output2 = persistent_app.invoke({"chat_history": ["Message 2"]}, config=config_1)
print(f"Agent says: {output2['chat_history'][-1]}")

# --- SESSION 3: A completely different user thread ---
print("\n--- RUN 3 (Different Thread) ---")
config_2 = {"configurable": {"thread_id": "john_session"}}
output3 = persistent_app.invoke({"user_name": "John", "chat_history": ["Hello"]}, config=config_2)
print(f"Agent says: {output3['chat_history'][-1]}")

--- RUN 1 ---
Agent says: Welcome back, Abhishek! You have sent 2 messages total.

--- RUN 2 (Same Thread) ---
Agent says: Welcome back, Abhishek! You have sent 4 messages total.

--- RUN 3 (Different Thread) ---
Agent says: Welcome back, John! You have sent 2 messages total.
